In [1]:
import os
import re
import numpy as np
from google import genai
from dotenv import load_dotenv

load_dotenv(override=True)

GEMINI_API_KEY = os.getenv("GEMINI_API_KEY", "YOUR_API_KEY_HERE")
client = genai.Client(api_key=GEMINI_API_KEY)

GEN_MODEL = "gemini-2.5-flash"
EMBED_MODEL = "models/gemini-embedding-001"  #  embedding model name

print("Client ready.")

Client ready.


In [2]:
documents = [
    {
        "source": "policy_handbook.md",
        "text": "Employees can work remotely up to three days per week. Security training is mandatory every quarter. VPN is required for accessing internal systems."
    },
    {
        "source": "it_support_faq.md",
        "text": "Password resets are available through the self-service portal. MFA enrollment is required for all accounts. For urgent incidents, contact the IT hotline."
    },
    {
        "source": "benefits_guide.md",
        "text": "Health insurance coverage starts on the first day of the month after joining. Annual wellness reimbursement is up to 300 USD. Paid parental leave is available for eligible employees."
    }
]

def chunk_text(text, chunk_size=90, overlap=20):
    text = text.strip()
    chunks = []
    start = 0
    while start < len(text):
        end = start + chunk_size
        chunks.append(text[start:end])
        if end >= len(text):
            break
        start = end - overlap
    return chunks

chunk_records = []
counter = 1
for doc in documents:
    for piece in chunk_text(doc["text"], chunk_size=90, overlap=20):
        cid = f"C{counter}"
        chunk_records.append({"chunk_id": cid, "source": doc["source"], "text": piece})
        counter += 1

print(f"Created {len(chunk_records)} chunks.")
for c in chunk_records:
    print(f"[{c['chunk_id']}] ({c['source']}) -> {c['text']}")

Created 7 chunks.
[C1] (policy_handbook.md) -> Employees can work remotely up to three days per week. Security training is mandatory ever
[C2] (policy_handbook.md) -> ng is mandatory every quarter. VPN is required for accessing internal systems.
[C3] (it_support_faq.md) -> Password resets are available through the self-service portal. MFA enrollment is required 
[C4] (it_support_faq.md) -> ollment is required for all accounts. For urgent incidents, contact the IT hotline.
[C5] (benefits_guide.md) -> Health insurance coverage starts on the first day of the month after joining. Annual welln
[C6] (benefits_guide.md) -> oining. Annual wellness reimbursement is up to 300 USD. Paid parental leave is available f
[C7] (benefits_guide.md) -> leave is available for eligible employees.


In [3]:
# Let's check what models are available
print("Available embedding models:")
for model in client.models.list():
    if 'embed' in model.name.lower():
        print(f"  - {model.name}")

Available embedding models:
  - models/gemini-embedding-001
  - models/gemini-embedding-2-preview
  - models/gemini-embedding-2


In [5]:
def embed_texts(texts, model=EMBED_MODEL):
    """Generate embeddings for a list of texts using the Gemini embedding model."""
    result = client.models.embed_content(contents=texts, model=model)

    #Extract embeddings from the response
    embeddings = []
    if hasattr(result, 'embeddings'):
        for emb in result.embeddings:
            if hasattr(emb, 'values'):
                embeddings.append(np.array(emb.values, dtype=np.float32))
    return embeddings

def cosine_similarity(a, b):
    """Calculate cosine similarity between two vectors (0=unrelated, 1=identical)."""
    denom=(np.linalg.norm(a) * np.linalg.norm(b))
    if denom == 0:
        return 0.0
    return float(np.dot(a, b) / denom)

chunk_texts = [c["text"] for c in chunk_records]
chunk_vectors = embed_texts(chunk_texts)

def retrieve(query, top_k=3):
    q_vec = embed_texts([query])[0]
    scored = []
    for rec, vec in zip(chunk_records, chunk_vectors):
        score = cosine_similarity(q_vec, vec)
        scored.append((score, rec))
    scored.sort(key=lambda x: x[0], reverse=True)
    return scored[:top_k]

query = "What are the rules for remote work and secure access?"
top_hits = retrieve(query, top_k=3)

print("Top retrieved chunks:")
for score, rec in top_hits:
    print(f"score={score:.4f} [{rec['chunk_id']}] ({rec['source']}) -> {rec['text']}")

Top retrieved chunks:
score=0.7660 [C1] (policy_handbook.md) -> Employees can work remotely up to three days per week. Security training is mandatory ever
score=0.6860 [C2] (policy_handbook.md) -> ng is mandatory every quarter. VPN is required for accessing internal systems.
score=0.6110 [C4] (it_support_faq.md) -> ollment is required for all accounts. For urgent incidents, contact the IT hotline.


In [6]:
def answer_with_citations(query, top_k=3):
    hits = retrieve(query, top_k=top_k)

    context_lines = []
    for _, rec in hits:
        context_lines.append(
            f"[{rec['chunk_id']}] source={rec['source']}\n{rec['text']}"
        )

    context_block = "\n\n".join(context_lines)

    prompt = f"""
You are a grounded assistant.
Answer ONLY using the context below.
If the context is insufficient, say: 'I don't have enough context.'
For every factual claim, include at least one citation tag like [C1].

Question:
{query}

Context:
{context_block}
"""

    resp = client.models.generate_content(
        model=GEN_MODEL,
        contents=prompt
    )

    answer = resp.text
    found_citations = sorted(set(re.findall(r"\[C\d+\]", answer)))

    return answer, hits, found_citations


final_answer, final_hits, cites = answer_with_citations(
    "How often is security training required and what is needed for internal system access?",
    top_k=3
)

print("Answer:")
print(final_answer)

print("\nCitations found in answer:", cites)

print("\nRetrieved evidence:")
for score, rec in final_hits:
    print(
        f" - [{rec['chunk_id']}] ({rec['source']}) score={score:.4f}"
    )

Answer:
Security training is mandatory every quarter [C1, C2]. For accessing internal systems, VPN is required [C2].

Citations found in answer: ['[C2]']

Retrieved evidence:
 - [C1] (policy_handbook.md) score=0.7376
 - [C2] (policy_handbook.md) score=0.6980
 - [C4] (it_support_faq.md) score=0.6131
